About this notebook

In [1]:
import numpy as np
import pandas as pd
import os
import operator
import matplotlib
import argparse
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from copy import deepcopy
import corner
from tqdm import tqdm
import matplotlib.lines as mlines
from scipy.stats import dirichlet
from scipy.stats import loguniform

from matplotlib import gridspec

import sys
sys.path.append('/data/wiay/2297403c/amaze_model_select/AMAZE_model_selection/')
from populations.bbh_models import *
from populations.Pop_Flows import FlowModel
from sample.sample import lnlike_disc

/data/wiay/2297403c/conda_envs/amaze/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
np.random.seed(12)
colors = sns.color_palette("colorblind", n_colors=10)
cp = [colors[0], colors[2], colors[4], colors[1], colors[3], colors[6], colors[9], colors[5], colors[8]]
#plt.style.use("mpl.sty")

_param_dict = {'mchirp' : {'limits':(0,100), 'fullname':'Chirp Mass [$M_\\odot$]'},
    'q' : {'limits':(0,1), 'fullname':'Mass Ratio'},
    'chieff' : {'limits':(-1,1), 'fullname':'Effective Inspiral Spin'},
    'z' : {'limits':(0,2), 'fullname':'Redshift'},
    }
_hyperparam_dict = {'chib' : {'values':{'chi00':0.0, 'chi01':0.1, 'chi02':0.2, 'chi05':0.5},
        'fullname':'$\\chi_\\mathrm{birth}$'},
    'alphaCE' : {'values':{'alpha02':np.log(0.2), 'alpha05':np.log(0.5), 'alpha10':np.log(1.0), 'alpha20':np.log(2.0), 'alpha50':np.log(5.0)},
        'fullname':'$\\alpha_\\mathrm{CE}$'}
    }
_channels_dict = {
    'CE':  {'parameters':['chib','alphaCE'], 'fullname':'Common Envelope'},
    'CHE': {'parameters':['chib'], 'fullname':'Chemically Homogeneous Evolution'},
    'GC':  {'parameters':['chib'], 'fullname':'Globular Clusters'},
    'NSC': {'parameters':['chib'], 'fullname':'Nuclear Star Clusters'},
    'SMT': {'parameters':['chib'], 'fullname':'Stable Mass Transfer'}
    }

_param_bounds = {"mchirp": (-1.,70), "q": (0.08,1.01), "chieff": (-0.6,1.), "z": (-0.1,3.5)}
_param_ticks = {"mchirp": [0,10,20,30,40,50,60,70], "q": [0.25,0.5,0.75,1], "chieff": [-0.5,0,0.5,1], "z": [0,0.25,0.5,0.75,1.0,1.25]}
_pdf_bounds = {"mchirp": (0,0.075), "q": (0,32), "chieff": (0,17), "z": (0,4)}
_pdf_ticks = {"mchirp": [0.0,0.025,0.050,0.075], "q": [0,10,20,30], "chieff": [0,4,8,12,16], "z": (0,1,2,3,4)}
_labels_dict = {"mchirp": r"$\mathcal{M}$/$M_{\odot}$", "q": r"$q$", \
"chieff": r"$\chi_{\rm eff}$", "z": r"$z$", "chi00": r"$\chi_\mathrm{b}=0.0$", \
"chi01": r"$\chi_\mathrm{b}=0.1$", "chi02": r"$\chi_\mathrm{b}=0.2$", \
"chi05": r"$\chi_\mathrm{b}=0.5$", "alpha02": r"$\alpha_\mathrm{CE}=0.2$", \
"alpha05": r"$\alpha_\mathrm{CE}=0.5$", "alpha10": r"$\alpha_\mathrm{CE}=1.0$", \
"alpha20": r"$\alpha_\mathrm{CE}=2.0$", "alpha50": r"$\alpha_\mathrm{CE}=5.0$", \
"CE": r"$\texttt{CE}$", "CHE": r"$\texttt{CHE}$", "GC": r"$\texttt{GC}$", \
"NSC": r"$\texttt{NSC}$", "SMT": r"$\texttt{SMT}$", \
"chi_b": r"$\chi_\mathrm{b}$", "alpha_CE": r"$\alpha_\mathrm{CE}$"}
_param_label = [_labels_dict["mchirp"],_labels_dict["q"], _labels_dict["chieff"], _labels_dict["z"]]
_Nsamps = 100000
_channel_label =[r'$\beta_{\mathrm{CE}}$',r'$\beta_{\mathrm{CHE}}$',r'$\beta_{\mathrm{GC}}$',r'$\beta_{\mathrm{NSC}}$',r'$\beta_{\mathrm{SMT}}$']
_channel_label_det =[r'$\beta_{\mathrm{CE}}^{\mathrm{det}}$',r'$\beta_{\mathrm{CHE}}^{\mathrm{det}}$',r'$\beta_{\mathrm{GC}}^{\mathrm{det}}$',\
    r'$\beta_{\mathrm{NSC}}^{\mathrm{det}}$',r'$\beta_{\mathrm{SMT}}^{\mathrm{det}}$']
_beta_det_label = r'$p(\beta^{\mathrm{det}})$'

_models_path ='/data/wiay/2297403c/models_reduced.hdf5'

pt = 1./72.27
jour_sizes = {"AAS": {"onecol": 242.26653*pt, "twocol": 242.26653*2*pt},
              # Add more journals below. Can add more properties to each journal
             }

figure_width = jour_sizes["AAS"]["onecol"]


_base_corner_kwargs = dict(
    bins=60,
    smooth=0.9,
    #quantiles=[0.16, 0.84],
    levels=(0.5,0.9,0.99),
    plot_density=True,
    plot_datapoints=True,
    fill_contours=True,
    show_titles=False,
    hist_kwargs=dict(density=True,linewidth=.75),
    contour_kwargs=dict(linewidths=1.),
    labels=_param_label,
    hist2d_kwargs= dict(data_kwargs=dict(alpha=0.01))
)

In [8]:
#Figure 1 - sampling from the trained normalising flow
def sample_pop_corner(popsynth_outputs, flow_dir, channel_label, hyperparam_idxs, effectiveNsamps=True, sample_KDE=True):

    print('loading flow and KDE models...')
    model_names, flow = get_models(_models_path, [channel_label], _param_dict, True, _hyperparam_dict, _channels_dict, sensitivity='midhighlatelow')
    if sample_KDE:
        KDE_models = get_models(_models_path, [channel_label], _param_dict, False, _hyperparam_dict, _channels_dict, sensitivity='midhighlatelow')

    flow[channel_label].load_model(flow_dir, device='cpu')

    if effectiveNsamps:
        #determine no. effective samples to draw from flow/KDE
        models_dict = dict.fromkeys(popsynth_outputs.keys())
        weights_dict = dict.fromkeys(popsynth_outputs.keys())

        for key in popsynth_outputs.keys():
            models_dict[key] = popsynth_outputs[key][_params]
            weights_dict[key]= popsynth_outputs[key]['weight']
        weights=weights_dict[tuple(hyperparam_idxs)]
        _Nsamps = int((np.sum(weights))**2/(np.sum(weights**2)))

    #sample flow
    print('sampling flow...')
    conditional = [flow[channel_label].hp_vals[i][hyperparam_idxs[i]] for i in range(smdl.conditionals)]
    flow_samples_stack = flow[channel_label].sample(np.array([conditional]),_Nsamps)

    if sample_KDE:
        print('sampling KDE...')
        model_key = [[channel_label]]
        for i,model_indx in enumerate(hyperparam_idxs):
            smdl_name=list(_hyperparam_dict[list(_hyperparam_dict.keys())[1]]['values'].keys())[model_indx]
            model_list.append(_hyperparam_dict[hyper_idx][int(np.floor(x[hyper_idx]))])

        kde_smdl = reduce(operator.getitem, model_key, KDE_models)
        kde_samples=kde_smdl.sample(_Nsamps)
        return fig,flow_samples,kde_samples
    else:
        return fig,flow_samples

def make_pop_corner(channel_label, hyperparam_idxs, flow_dir, conditional=None, plot_KDE=True, outdir='.', testCE=False):

    #get training population samples
    popsynth_outputs = read_hdf5(_models_path, channel_label) # read all data from hdf5 file
    models_dict = dict.fromkeys(popsynth_outputs.keys())
    weights_dict = dict.fromkeys(popsynth_outputs.keys())

    for key in popsynth_outputs.keys():
        models_dict[key] = popsynth_outputs[key][_params]
        weights_dict[key]= popsynth_outputs[key]['weight']

    #get samples from models
    sample_pop_corner(popsynth_outputs, flow_dir, channel_label, hyperparam_idxs=hyperparam_idxs, sample_KDE=plot_KDE)

    #set colours for plot
    if testCE:
        colors=['C1', 'royalblue']
        labels=['Underlying Model', 'Normalising Flow']
    else:
        colors=['C1', 'purple', 'royalblue']
        labels=['Underlying Model', 'KDE', 'Normalising Flow']

    model_kwargs = deepcopy(_base_corner_kwargs)
    model_kwargs["color"] = colors[0]
    model_kwargs["hist_kwargs"]["color"] = colors[0]
    corner_kwargs_kde = deepcopy(_base_corner_kwargs)
    corner_kwargs_kde["color"] = colors[1]
    corner_kwargs_kde["hist_kwargs"]["color"] = colors[1]
    corner_kwargs_flow = deepcopy(_base_corner_kwargs)
    corner_kwargs_flow["color"] = colors[2]
    corner_kwargs_flow["hist_kwargs"]["color"] = colors[2]

    print('plotting samples')
    #else plot training models, KDE if plotting, and flow last
    if hyperparam_idxs.ndim > 0:
        dict_key = tuple(hyperparam_idxs)
    else:
        dict_key = hyperparam_idxs
    fig =corner.corner(models_dict[dict_key],  weights=weights_dict[dict_key], **model_kwargs)
    if plot_KDE==True:
        corner.corner(kde_samples, fig=fig, **corner_kwargs_kde)
    corner.corner(flow_samples, fig=fig, **corner_kwargs_flow)
    #add legend
    plt.legend(
            handles=[
                mlines.Line2D([], [], color=colors[i], label=labels[i])
                for i in range(len(colors))
            ],
            frameon=False,
            bbox_to_anchor=(1, 4), loc="upper right"
        )
    #save figure
    fig.set_size_inches(figure_width*1.5, figure_width*1.5)
    """if testCE:
        plt.savefig(f"{outdir}/pdfs/{channel_label}_flowKDEmodel_corner_chib{hyperparam_idxs[0]}testCE.pdf")
    else:
        plt.savefig(f"{outdir}/pdfs/{channel_label}_flowKDEmodel_corner_chib{hyperparam_idxs[0]}.pdf")"""
    return fig, flow_samples, kde_samples

In [9]:
make_pop_corner('CE',[0,1], flow_dir='/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/inputs/flow_models/mixed_models/')

TypeError: read_hdf5() missing 2 required positional arguments: 'channel_smdl_names' and 'smdl_indxs_combos'

In [ ]:
#Fig 2 Test CE plot
make_pop_corner('CE',[1,2], flow_dir='/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/inputs/flow_models/CE_test_population/', plot_KDE=False)

In [ ]:
#Fig 3. Log likelihood ratio plot

def calc_llh_ratio_CE(flow_dir, outdir=_basepath):

    channel_label = 'CE'
    popsynth_outputs = read_hdf5(_models_path, channel_label) # read all data from hdf5 file

    model_names, flow = get_models(_models_path, [channel_label], _params, use_flows=True, \
         detectable=False, senseitivity='midhighlatelow_network', flow_path=flow_dir, device='cpu')
    model_names, KDE_models = get_models(_models_path, [channel_label], _params, use_flows=False, normalize=False,\
         detectable=False, senseitivity='midhighlatelow_network')

    hyperparams = list(set([x.split('/', 1)[1] for x in model_names]))
    Nhyper = np.max([len(x.split('/')) for x in hyperparams])

    # construct dict that relates submodels to their index number
    submodels_dict = {} #dummy index dict keys:0,1,2,3, items: particular models
    ctr=0 #associates with either chi_b or alpha (0 or 1)
    while ctr < Nhyper:
        submodels_dict[ctr] = {}
        hyper_set = sorted(list(set([x.split('/')[ctr] for x in hyperparams])))
        for idx, model in enumerate(hyper_set): #idx associates with 0,1,2,3,(4) keys
            submodels_dict[ctr][idx] = model
        ctr += 1

    flow[channel_label].load_model(flow_dir, device='cpu')

    mchirps = np.linspace(4.,49.9,20)
    qs = np.linspace(0.01,0.99,20)

    p_mchirpq_unreg = np.zeros((20,20))
    p_mchirpq_kde_unreg = np.zeros((20,20))
    p_mchirpq_reg = np.zeros((20,20))
    p_mchirpq_kde_reg = np.zeros((20,20))
    chi_b_id = 0
    alpha_id = 4

    for  i, m in enumerate(tqdm(mchirps)):
        for j, q in enumerate(qs):
            sample = np.reshape([m, q,0.05,0.2], (1,1,4))
            p_mchirpq_reg[i, j] = lnlike_disc([chi_b_id,alpha_id], sample, flow, submodels_dict, ['CE'], use_flows=True,\
                 prior_pdf=None, smallest_N=990903)
            p_mchirpq_kde_reg[i, j] = lnlike_disc([chi_b_id,alpha_id], sample, KDE_models, submodels_dict, ['CE'], use_flows=False,\
                 prior_pdf=None, smallest_N=990903)
            
            p_mchirpq_unreg[i, j] = lnlike_disc([chi_b_id,alpha_id], sample, flow, submodels_dict, ['CE'], use_flows=True,\
                 prior_pdf=None, smallest_N=None)
            p_mchirpq_kde_unreg[i, j] = lnlike_disc([chi_b_id,alpha_id], sample, KDE_models, submodels_dict, ['CE'], use_flows=False,\
                 prior_pdf=None, smallest_N=None)

    llh_ratio_kde_flow_reg = np.log10(np.exp(p_mchirpq_reg-p_mchirpq_kde_reg))
    llh_ratio_kde_flow_unreg = np.log10(np.exp(p_mchirpq_unreg-p_mchirpq_kde_unreg))

    #save ratios
    np.save(f"{outdir}/data/llh_ratio_kde_flow_reg.npy", llh_ratio_kde_flow_reg)
    np.save(f"{outdir}/data/llh_ratio_kde_flow_unreg.npy", llh_ratio_kde_flow_unreg)

    return mchirps, qs, llh_ratio_kde_flow_reg, llh_ratio_kde_flow_unreg

def plot_llh_ratio_CE(flow_dir, outdir, justplot=False):
    plt.rcParams['figure.figsize'] = [figure_width, figure_width*1.2]
    channel_label = 'CE'
    
    if justplot:
        mchirps = np.linspace(4.,49.9,20)
        qs = np.linspace(0.01,0.99,20)
        llh_ratio_kde_flow_reg = np.load(f"{outdir}/data/llh_ratio_kde_flow_reg.npy")
        llh_ratio_kde_flow_unreg = np.load(f"{outdir}/data/llh_ratio_kde_flow_unreg.npy")
    else:
        mchirps, qs, llh_ratio_kde_flow_reg, llh_ratio_kde_flow_unreg = calc_llh_ratio_CE(flow_dir, outdir)

    popsynth_outputs = read_hdf5(_models_path, channel_label)
    models_dict = dict.fromkeys(popsynth_outputs.keys())
    weights_dict = dict.fromkeys(popsynth_outputs.keys())

    for key in popsynth_outputs.keys():
        models_dict[key] = popsynth_outputs[key][_params]
        weights_dict[key]= popsynth_outputs[key]['weight']

    chi_b_id = 0
    alpha_id = 4
    
    fig, ax = plt.subplots(2,1)
    #cbar_scales = [50,2]

    for row, ratio in enumerate([llh_ratio_kde_flow_unreg, llh_ratio_kde_flow_reg]):
        #cbar_scale= cbar_scales[row]
        cbar_scale = np.max(np.abs(ratio))
        c = ax[row].imshow(np.swapaxes(ratio, 0,1), extent=(mchirps[0], mchirps[-1], qs[0], qs[-1]), origin='lower',\
            vmin=-cbar_scale, vmax=cbar_scale, aspect='auto', cmap='RdBu')
        cbar = fig.colorbar(c, ax=ax[row], cmap='RdBu')
        cbar.set_label(r'log$_{10}$ (p$_\mathrm{flow}/$p$_\mathrm{KDE}$)')

        ax[row].set_ylabel(_labels_dict['q'])

        bin_width = 0.05
        chieffs = popsynth_outputs[(chi_b_id,alpha_id)][:]['chieff']
        zs = popsynth_outputs[(chi_b_id,alpha_id)][:]['z']
        bin_chieff = np.logical_and(chieffs>0.0 - 2*bin_width,  chieffs< 0.0 + 2*bin_width)
        bin_z = np.logical_and(zs>0.2 - 10*bin_width, zs < 0.2 + 10*bin_width)
        bin_conditions = np.logical_and(bin_chieff, bin_z)

        c = ax[row].imshow(np.swapaxes(ratio,0,1), extent=(mchirps[0], mchirps[-1], qs[0], qs[-1]), origin='lower',\
            vmin=-cbar_scale, vmax=cbar_scale, aspect='auto', zorder=-200, cmap='RdBu')

        corner.hist2d(np.array(popsynth_outputs[(chi_b_id,alpha_id)][bin_conditions]['mchirp']),\
            np.array(popsynth_outputs[(chi_b_id,alpha_id)][bin_conditions]['q']), bins =16, \
            levels=(.50, .90, .99), \
            weights=np.array(weights_dict[(chi_b_id,alpha_id)][bin_conditions]), contour_kwargs=dict(linewidths=.5), \
            pcolor_kwargs=dict(alpha=0.0), density=True, ax=ax[row], no_fill_contours=True,\
            plot_datapoints=False)
        ax[row].set_xlim(mchirps[0], mchirps[-1])
        ax[row].set_ylim(qs[0], qs[-1])
        ax[1].set_xlabel(_labels_dict['mchirp'])


    fig.tight_layout(pad=1.3)
    fig.savefig(f'{outdir}/pdfs/CE2Dmchirpq_llhratio.pdf')

In [ ]:
#Fig 4. Discrete Result

In [ ]:
#Fig 5. Continuous result

In [ ]:
#Fig. 6 Dataspace result

In [ ]:
# KLs